# 02. Feature Engineering Engine
## Transforming Raw DNS Queries into Discriminative ML Vectors

This notebook demonstrates the feature engineering pipeline implemented in `src/features.py`:
1. **Information Theory**: Shannon Entropy (full domain, subdomain, individual labels).
2. **Lexical & Structural Metrics**: Domain lengths, label counts, min/max/avg label lengths.
3. **Linguistic Ratios**: Vowel ratio, digit ratio, consonant ratio, vowel-to-consonant, hex ratio.
4. **Temporal Frequency**: Rolling query frequency (queries per minute/second) for burst vs slow attacks.
5. **Batch Processing**: Transforming raw PCAP captures or labeled CSVs into feature matrices ready for ML.

In [1]:
import os
import sys
import yaml
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")

# Ensure repo root is on sys.path
for p in [".", "..", "../.."]:
    if os.path.exists(os.path.join(p, "configs", "config.yaml")):
        sys.path.insert(0, os.path.abspath(p))
        REPO_ROOT = os.path.abspath(p)
        break

# Load configuration
config_path = os.path.join(REPO_ROOT, "configs", "config.yaml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

from src.features import (
    shannon_entropy,
    compute_entropy,
    vowel_digit_ratio,
    max_consecutive_consonants,
    split_domain_components,
    extract_domain_features,
    calculate_query_frequency,
    process_traffic_dataset
)

print("Feature engineering modules loaded successfully.")

Feature engineering modules loaded successfully.


### 1. Information-Theoretic Metrics: Shannon Entropy
Shannon entropy quantifies the uncertainty or randomness of characters in a string:

$$H(X) = - \sum_{i=1}^n P(x_i) \log_2 P(x_i)$$

Natural domain names (e.g. `google.com`, `wikipedia.org`) have repetitive linguistic patterns resulting in low entropy.
Base32 and Hex encoded DNS tunnels (e.g. `54657374...tunnel.org`) exhibit near-maximal entropy for their character sets.

In [2]:
# Comparing entropy across benign, Base32, and Hex queries
sample_queries = {
    "Benign - Standard": "google.com",
    "Benign - Subdomain": "cdn.assets.cloudflare.net",
    "Exfil - Base32": "sq0001.krsxg5bamnxw4ztjmrsw45djmfwcazdb.tunnel.example.com",
    "Exfil - Hex": "sq0001.5465737420636f6e666964656e7469616c.tunnel.example.com"
}

for desc, q in sample_queries.items():
    ent = compute_entropy(q)
    parts = split_domain_components(q)
    sub_ent = compute_entropy(parts["subdomain"]) if parts["subdomain"] else 0.0
    print(f"{desc:20} | FQDN: {q[:45]:45}... | Total H: {ent:.4f} | Subdomain H: {sub_ent:.4f}")

Benign - Standard    | FQDN: google.com                                   ... | Total H: 2.6464 | Subdomain H: 0.0000
Benign - Subdomain   | FQDN: cdn.assets.cloudflare.net                    ... | Total H: 3.5933 | Subdomain H: 2.8464
Exfil - Base32       | FQDN: sq0001.krsxg5bamnxw4ztjmrsw45djmfwcazdb.tunne... | Total H: 4.6012 | Subdomain H: 4.5285
Exfil - Hex          | FQDN: sq0001.5465737420636f6e666964656e7469616c.tun... | Total H: 4.1198 | Subdomain H: 3.7439


### 2. Lexical & Structural Metrics: Label Lengths & Components
According to RFC 1035:
- Labels must not exceed 63 characters.
- FQDN must not exceed 253 characters.

Attackers maximize each label to fit up to 63 bytes of payload, leading to unusually high `max_label_len` and `subdomain_len`.

In [3]:
# Extracting component and length features
demo_features = []
for desc, q in sample_queries.items():
    feats = extract_domain_features(q)
    feats["description"] = desc
    demo_features.append(feats)

df_demo = pd.DataFrame(demo_features)
cols = ["description", "domain_len", "subdomain_len", "num_labels", "max_label_len", "avg_label_len", "entropy"]
df_demo[cols]

,description,domain_len,subdomain_len,num_labels,max_label_len,avg_label_len,entropy
0,Benign - Standard,10,0,2,6,4.5,2.6464
1,Benign - Subdomain,25,10,4,10,5.5,3.5933
2,Exfil - Base32,58,46,5,32,10.8,4.6012
3,Exfil - Hex,60,48,5,34,11.2,4.1198


### 3. Linguistic Ratios: Vowel/Digit Ratios & Consonants
Normal English / human words have regular vowel-to-consonant ratios (~0.4 - 0.7).
Base32 has few vowels (`A, E, I`) relative to consonants, and Hex contains only `A-F` with `0-9` digits.

In [4]:
# Inspecting ratio metrics
ratio_cols = ["description", "digit_ratio", "hex_ratio", "vowel_ratio", "consonant_ratio", "max_consecutive_consonants"]
df_demo[ratio_cols]

,description,digit_ratio,hex_ratio,vowel_ratio,consonant_ratio,max_consecutive_consonants
0,Benign - Standard,0.0000,0.2000,0.4444,0.5556,2
1,Benign - Subdomain,0.0000,0.4000,0.3182,0.6818,3
2,Exfil - Base32,0.1379,0.3621,0.1739,0.8261,7
3,Exfil - Hex,0.5667,0.7167,0.3636,0.6364,3


### 4. Temporal Frequency Metrics: Burst vs Low-and-Slow
DNS exfiltration attacks follow distinct operational profiles:
- **Burst mode**: Hundreds of queries sent in milliseconds/seconds to exfiltrate data quickly.
- **Low-and-Slow mode**: Covert queries sent at intervals (every 10-60s) to evade volume-based alarms.

In [5]:
# Demonstrating rolling query frequency computation
time_records = pd.DataFrame({
    "qname": ["tunnel.exfil.org"] * 5 + ["google.com"] * 2,
    "timestamp": [1710000000.0, 1710000001.0, 1710000002.0, 1710000002.5, 1710000003.0, 1710000000.0, 1710000030.0]
})

freq_result = calculate_query_frequency(time_records, window="10s", rate_unit="second")
freq_result

,qname,timestamp,queries_per_second
0,google.com,1.710000e+09,0.1
1,google.com,1.710000e+09,0.1
2,tunnel.exfil.org,1.710000e+09,0.1
3,tunnel.exfil.org,1.710000e+09,0.2
4,tunnel.exfil.org,1.710000e+09,0.3
5,tunnel.exfil.org,1.710000e+09,0.4
6,tunnel.exfil.org,1.710000e+09,0.5


### 5. Full Pipeline: Processing Raw Traffic into ML Feature Dataset

In [6]:
# Process existing labeled traffic dataset or synthetic session
raw_traffic_path = os.path.join(REPO_ROOT, config["paths"].get("labeled_traffic_csv", "data/processed/labeled_traffic.csv"))
output_features_path = os.path.join(REPO_ROOT, config["paths"].get("features_dataset_csv", "data/processed/dns_features.csv"))

if os.path.exists(raw_traffic_path):
    print(f"Loading raw traffic from {raw_traffic_path}...")
    df_raw = pd.read_csv(raw_traffic_path)
else:
    print("Synthesizing raw traffic session...")
    from src.capture_traffic import generate_synthesized_traffic
    records, _ = generate_synthesized_traffic(benign_count=200, burst_count=120, low_slow_count=40)
    df_raw = pd.DataFrame(records)

# Execute feature extraction pipeline
df_features = process_traffic_dataset(df_raw, window="60s")
os.makedirs(os.path.dirname(output_features_path), exist_ok=True)
df_features.to_csv(output_features_path, index=False)

print(f"Successfully processed {len(df_features)} records into feature matrix.")
print(f"Saved feature dataset to: {output_features_path}")
df_features.head()

Loading raw traffic from /home/hyperghost/github/dns-exfiltration-rm/data/processed/labeled_traffic.csv...
Successfully processed 115 records into feature matrix.
Saved feature dataset to: /home/hyperghost/github/dns-exfiltration-rm/data/processed/dns_features.csv


,domain_len,subdomain_len,num_labels,max_label_len,min_label_len,avg_label_len,entropy,subdomain_entropy,max_label_entropy,digit_ratio,...,vowel_consonant_ratio,special_ratio,subdomain_digit_ratio,subdomain_hex_ratio,unique_char_ratio,max_consecutive_consonants,domain,label,attack_type,queries_per_minute
0,9,0,2,5,3,4.00,2.9477,0.0000,2.3219,0.0000,...,0.1429,0.0000,0.0,0.0,0.8889,5,fbcdn.net,0,benign,1.0
1,6,0,2,3,2,2.50,1.9183,0.0000,1.0000,0.0000,...,0.6667,0.0000,0.0,0.0,0.6667,2,goo.gl,0,benign,1.0
2,18,5,3,8,3,5.33,3.4613,2.3219,2.5000,0.0556,...,0.4000,0.0556,0.2,0.6,0.6667,3,cdn-4.linkedin.com,0,benign,1.0
3,17,5,3,7,3,5.00,3.6169,2.3219,2.3219,0.0588,...,0.4444,0.0588,0.2,0.6,0.7647,3,cdn-9.reuters.com,0,benign,1.0
4,10,0,2,7,2,4.50,2.9219,0.0000,2.5216,0.0000,...,0.2857,0.0000,0.0,0.0,0.8000,2,discord.gg,0,benign,1.0


In [7]:
# Feature Summary & Correlation with Target
numeric_features = [c for c in df_features.select_dtypes(include=[np.number]).columns if c not in ["label"]]
print(f"Total numeric features generated: {len(numeric_features)}")
print(numeric_features)

if "label" in df_features.columns:
    corr = df_features[numeric_features].apply(lambda x: x.corr(df_features["label"])).sort_values(ascending=False)
    print("\nFeature correlation with malicious label:")
    print(corr.round(4).to_string())

Total numeric features generated: 21
['domain_len', 'subdomain_len', 'num_labels', 'max_label_len', 'min_label_len', 'avg_label_len', 'entropy', 'subdomain_entropy', 'max_label_entropy', 'digit_ratio', 'alpha_ratio', 'hex_ratio', 'vowel_ratio', 'consonant_ratio', 'vowel_consonant_ratio', 'special_ratio', 'subdomain_digit_ratio', 'subdomain_hex_ratio', 'unique_char_ratio', 'max_consecutive_consonants', 'queries_per_minute']

Feature correlation with malicious label:
subdomain_len                 0.9991
domain_len                    0.9964
max_label_len                 0.9961
avg_label_len                 0.9827
num_labels                    0.9737
subdomain_entropy             0.9259
entropy                       0.9135
digit_ratio                   0.8786
max_label_entropy             0.8659
subdomain_digit_ratio         0.8427
subdomain_hex_ratio           0.7599
hex_ratio                     0.7438
max_consecutive_consonants    0.3548
min_label_len                 0.3114
consonant_ra